In [1]:
import pandas as pd
import tqdm.notebook as tqdm

import re

In [2]:
df = pd.read_excel("LLM_outputs.xlsx")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10584 entries, 0 to 10583
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Unnamed: 0                     10584 non-null  int64 
 1   id                             10584 non-null  int64 
 2   company                        10584 non-null  object
 3   job title                      10584 non-null  object
 4   text                           10582 non-null  object
 5   triples_qwen_structured        10584 non-null  object
 6   triples_qwen_semi-structured   10584 non-null  object
 7   triples_qwen_unstructured      10584 non-null  object
 8   triples_gemma_structured       10584 non-null  object
 9   triples_gemma_semi-structured  10584 non-null  object
 10  triples_gemma_unstructured     10584 non-null  object
 11  triples_llama_structured       10584 non-null  object
 12  triples_llama_semi-structured  10584 non-null  object
 13  t

In [4]:
def extract_triples(input_string):
    """
    1. Converts '*' to '"'.
    2. Extracts only 3-element tuples, accepting single or double quotes.
    3. Filters out noise and tuples of other lengths.
    """
    
    # --- Step 1: Pre-process the string ---
    # Replace single asterisks (*) with double quotes (")
    # This assumes asterisks are only used as quote delimiters.
    processed_string = input_string.replace('*', '"')

    # --- Step 2: Define the flexible Regex Pattern ---
    
    # Quoting Group (Q): This group (r'["\']') matches either a double quote OR a single quote.
    Q = r'["\']' 
    
    # String Content: This group (r'.*?') matches any character non-greedily inside the quotes.
    # The string must be captured by group (r'({Q}.*?{Q})')
    
    # Flexible Pattern (using f-string for clarity and Q definition):
    pattern = rf"""
        \(              # Match the literal opening parenthesis (
        ({Q}.*?{Q})     # Group 1: Capture the first quoted string
        ,\s* # Match comma, optional whitespace
        ({Q}.*?{Q})     # Group 2: Capture the second quoted string
        ,\s* # Match comma, optional whitespace
        ({Q}.*?{Q})     # Group 3: Capture the third quoted string
        \)              # Match the literal closing parenthesis )
    """
    
    # Use re.findall with re.VERBOSE for multiline pattern and re.DOTALL to match across newlines
    matches = re.findall(pattern, processed_string, re.VERBOSE | re.DOTALL)
    
    # --- Step 3: Clean up and format the final list ---
    extracted_data = []
    for str1_quoted, str2_quoted, str3_quoted in matches:
        # Remove the surrounding quotes from each captured string
        # using the replace method, which handles both ' and "
        str1 = str1_quoted.strip().replace('"', '').replace("'", '')
        str2 = str2_quoted.strip().replace('"', '').replace("'", '')
        str3 = str3_quoted.strip().replace('"', '').replace("'", '')
        extracted_data.append((str1, str2, str3))
        
    return extracted_data

In [10]:
test = df['triples_qwen_unstructured'].apply(extract_triples)

In [12]:
test.iloc[10583]

[('Job',
  'hasTitle',
  'Produktionsteknolog/teknisk designer eller tilsvarende 3D-konstruktør'),
 ('Job', 'hasLocation', 'København'),
 ('Job', 'requires', 'knowledgeOf3DDesign'),
 ('Job', 'requires', 'experienceWithCADSoftware'),
 ('Job', 'requires', 'proficiencyIn3DCADTools'),
 ('Job', 'requires', 'understandingOfProductionProcesses'),
 ('Job', 'requires', 'abilityToCreateTechnicalDrawings'),
 ('Job', 'requires', 'skillsInModelingForManufacturing'),
 ('Job', 'requires', 'familiarityWithMaterialProperties'),
 ('Job', 'requires', 'knowledgeOfTolerancesAndFit'),
 ('Job', 'requires', 'abilityToInterpretTechnicalSpecifications'),
 ('Job', 'requires', 'collaborationWithProductionTeams'),
 ('Job', 'requires', 'communicationSkillsForCrossFunctionalTeams'),
 ('Job', 'requires', 'attentionToDetail'),
 ('Job', 'requires', 'problemSolvingAbilities'),
 ('Job', 'requires', 'technicalProblemSolving'),
 ('Job', 'requires', 'projectPlanningSkills'),
 ('Job', 'requires', 'workWithBlueprintsAndSchema

In [6]:
extract_triples(df['triples_llama_unstructured'].iloc[10583])

[]

In [9]:
df['triples_llama_unstructured'].iloc[2391]

'Here are the triples for the "Store Manager" job listing:\n\n[""Job Title"" "isA" "Store Manager""]\n[""Location"" "isLocatedIn" "Denmark""]\n[""Industry"" "isIn" "Retail""]\n[""Job Type"" "is" "Full-time""]\n[""Salary Range"" "isBetween" "40000 - 60000" "DKK" "per month""]\n[""Experience" "requires" "2-3 years""]\n[""Education" "requires" "Bachelor\'s degree""]\n[""Language" "speaks" "Danish"" "English""]\n[""Skills" "includes" "Leadership"" "Communication"" "Team management""]\n[""Responsibilities" "includes" "Managing a team"" "Coordinating with suppliers"" "Maintaining store operations""]\n[""Requirements" "includes" "Strong leadership skills"" "Excellent communication skills"" "Ability to work under pressure""]\n[""We are looking for someone" "with" "proven experience in retail management"" "who has a strong background in leadership""]\n[""Ideal candidate" "has" "a degree in business or a related field""]\n[""We offer" "a competitive salary" "opportunities for career growth"" "a 